# Chunk, Embed & Knowledge-Graph Pipeline

Single end-to-end langgraph pipeline that:

1. Chunks `processed_notes_with_descriptions.md` with `MarkdownHeaderTextSplitter` (h1-h3) + `RecursiveCharacterTextSplitter` (1000/100).
2. Embeds each chunk with `BAAI/bge-m3` (cosine-normalized) and attaches the vector + metadata to the chunk.
3. Persists chunks to `chunks_with_embeddings.pkl` and stores them as `(:Chunk)` nodes in Neo4j.
4. Extracts educational nodes (`Topic`, `Subtopic`, `Concept`, `Definition`, `Formula`, `Theorem`, `Example`)
   and relationships (`COVERS`, `EXPLAINS`, `PREREQUISITE_FOR`, `PART_OF`, `USES_FORMULA`, `ILLUSTRATES`)
   using the custom Ollama transformer from `graph/langchain_final.ipynb`.
5. Links every entity node to the `(:Chunk)` nodes that mention it via `MENTIONED_IN`.

Environment notes: embeddings run on CPU (no CUDA detected), bge-m3 is already cached,
Neo4j is expected on `bolt://localhost:7687` and Ollama (`rnj-1:latest`) on `localhost:11434`.

# 1. Configuration

In [ ]:
import os
import re
import pickle
import hashlib

# Source markdown + models
MD_PATH = "/home/kanshu/poc-scripts/scripts/generated_107_notusellm/processed_notes_with_descriptions.md"
EMBED_MODEL = "BAAI/bge-m3"
EXTRACT_MODEL = "rnj-1:latest"
CONTEXT_WINDOW = 4096          # Ollama context window in tokens
VECTOR_DIM = 1024              # bge-m3 embedding size

# Neo4j connection
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "neo4jadmin"
NEO4J_DATABASE = "neo4j"

# Outputs
OUTPUT_DIR = "/home/kanshu/poc-scripts/scripts/embeddings_output"
CHUNKS_PKL = os.path.join(OUTPUT_DIR, "chunks_with_embeddings.pkl")
GRAPH_DOCS_PKL = "/home/kanshu/poc-scripts/graph/extracted_graph_docs.pkl"

# Runtime options
# Set to True to skip the (slow) Ollama extraction and reuse a previously
# persisted extracted_graph_docs.pkl. Default False = run the full pipeline.
LOAD_EXISTING_GRAPH_DOCS = False

# Ontology
KG_LABELS = ["Topic", "Subtopic", "Concept", "Definition", "Formula", "Theorem", "Example"]
ALLOWED_RELATIONSHIPS = ["COVERS", "EXPLAINS", "PREREQUISITE_FOR", "PART_OF", "USES_FORMULA", "ILLUSTRATES"]

IMAGE_PATTERN = re.compile(r"!\[.*?\]\((.*?)\)")


def chunk_id(text: str) -> str:
    """Stable md5 chunk identifier (matches build_graphrag_index.py)."""
    return hashlib.md5(text.encode("utf-8")).hexdigest()


def get_header(metadata: dict) -> str:
    """Pick the most specific markdown header available."""
    for key in ("Header 1", "Header 2", "Header 3"):
        if metadata.get(key):
            return metadata[key]
    return ""

print("Config loaded.")
print(f"  MD_PATH        = {MD_PATH}")
print(f"  EMBED_MODEL    = {EMBED_MODEL}")
print(f"  EXTRACT_MODEL  = {EXTRACT_MODEL}")
print(f"  CHUNKS_PKL     = {CHUNKS_PKL}")
print(f"  GRAPH_DOCS_PKL = {GRAPH_DOCS_PKL}")

# 2. Workflow state

In [ ]:
from typing_extensions import TypedDict


class GraphState(TypedDict):
    file_path: str                # markdown file to ingest
    raw_text: str                 # raw markdown contents
    chunks: list                  # Document chunks (with image metadata)
    embedded_chunks: list         # chunks after bge-m3 embedding attached
    extracted_graph_docs: list    # GraphDocuments from the Ollama extractor
    stats: dict                   # misc counters for printing

print("GraphState defined.")

# 3. Langgraph nodes

Each node is a small, self-contained function with print statements so you can
follow the pipeline step-by-step.

### 3.1 Wipe Neo4j (clean slate)

In [ ]:
def wipe_neo4j_node(state: GraphState) -> dict:
    """Delete previously stored KG entity nodes and Chunk nodes (scoped wipe)."""
    from neo4j import GraphDatabase

    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    deleted = 0
    try:
        with driver.session(database=NEO4J_DATABASE) as s:
            labels_pred = " OR ".join(f"n:{lbl}" for lbl in KG_LABELS + ["Chunk"])
            record = s.run(
                f"MATCH (n) WHERE {labels_pred} DETACH DELETE n RETURN count(n) AS deleted"
            ).single()
            deleted = record["deleted"] if record else 0
            s.run("DROP INDEX chunk_embeddings IF EXISTS").consume()
    finally:
        driver.close()
    print(f"[wipe_neo4j] Deleted {deleted} KG/Chunk node(s) from database '{NEO4J_DATABASE}'.")
    print("[wipe_neo4j] Dropped vector index 'chunk_embeddings' (recreated later).")
    return {"stats": {"deleted_nodes": deleted}}

### 3.2 Load markdown

In [ ]:
def load_markdown_node(state: GraphState) -> dict:
    """Reads the educational markdown chapter from disk."""
    with open(state["file_path"], "r", encoding="utf-8") as f:
        raw_text = f.read()
    print(f"[load_markdown] Read {len(raw_text)} characters from {state['file_path']}")
    return {"raw_text": raw_text}

### 3.3 Chunk markdown (headers + recursive splitter)

In [ ]:
def chunk_markdown_node(state: GraphState) -> dict:
    """Split raw text into structured markdown chunks, same logic as the embed script."""
    from langchain_core.documents import Document
    from langchain_text_splitters import (
        MarkdownHeaderTextSplitter,
        RecursiveCharacterTextSplitter,
    )

    raw_text = state["raw_text"]

    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "Header 1"),
            ("##", "Header 2"),
            ("###", "Header 3"),
        ],
        strip_headers=False,
    )
    md_header_splits = markdown_splitter.split_text(raw_text)
    print(f"[chunk_markdown] {len(md_header_splits)} header sections from MarkdownHeaderTextSplitter")

    # Detect images and attach metadata per header section
    docs_with_image_metadata = []
    for doc in md_header_splits:
        image_matches = IMAGE_PATTERN.findall(doc.page_content)
        metadata = doc.metadata.copy()
        metadata["images"] = image_matches if image_matches else []
        metadata["has_images"] = len(image_matches) > 0
        docs_with_image_metadata.append(
            Document(page_content=doc.page_content, metadata=metadata)
        )

    # Chunk large sections (bge-m3 supports up to 8192 tokens)
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    chunks = text_splitter.split_documents(docs_with_image_metadata)

    # Refine per-chunk metadata (scalar types for Neo4j) + assign ids
    for i, doc in enumerate(chunks):
        found_images = IMAGE_PATTERN.findall(doc.page_content)
        doc.metadata["images"] = ", ".join(found_images) if found_images else "none"
        doc.metadata["has_images"] = len(found_images) > 0
        doc.metadata["chunk_id"] = chunk_id(doc.page_content)
        doc.metadata["chunk_index"] = i
        doc.metadata["header"] = get_header(doc.metadata)
        doc.metadata["source"] = os.path.basename(state["file_path"])

    # Link the chunk chain (in document order): previous/next chunk ids.
    for i, doc in enumerate(chunks):
        doc.metadata["previous_chunk_id"] = (
            chunks[i - 1].metadata["chunk_id"] if i > 0 else None
        )
        doc.metadata["next_chunk_id"] = (
            chunks[i + 1].metadata["chunk_id"] if i < len(chunks) - 1 else None
        )

    print(f"[chunk_markdown] Created {len(chunks)} chunks.")
    print(f"[chunk_markdown] Sample metadata: {chunks[0].metadata}")
    return {"chunks": chunks, "stats": {"num_chunks": len(chunks)}}

### 3.4 Create embeddings (BAAI/bge-m3)

In [ ]:
def create_embeddings_node(state: GraphState) -> dict:
    """Embed every chunk with bge-m3 and attach the vector to doc.metadata['embedding']."""
    import torch
    from langchain_huggingface import HuggingFaceEmbeddings

    device = "cuda" if torch.cuda.is_available() else "cpu"
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBED_MODEL,
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": True},  # recommended for cosine similarity
    )

    texts = [doc.page_content for doc in state["chunks"]]
    print(f"[create_embeddings] Embedding {len(texts)} chunks with {EMBED_MODEL} on device='{device}' ...")
    vectors = embeddings.embed_documents(texts)

    for doc, vec in zip(state["chunks"], vectors):
        doc.metadata["embedding"] = vec

    dim = len(vectors[0]) if vectors else 0
    print(f"[create_embeddings] Done: {len(vectors)} vectors x {dim} dims (cosine-normalized).")
    print("[create_embeddings] Embedding stored under doc.metadata['embedding'].")
    return {"embedded_chunks": state["chunks"], "stats": {"embed_dim": dim}}

### 3.5 Save chunks (with embeddings + metadata) to .pkl

In [ ]:
def save_chunks_to_pkl_node(state: GraphState) -> dict:
    """Persist chunks (text + metadata + embedding) so other scripts can reuse them."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(CHUNKS_PKL, "wb") as f:
        pickle.dump(state["embedded_chunks"], f)
    print(f"[save_chunks_to_pkl] Saved {len(state['embedded_chunks'])} chunks to {CHUNKS_PKL}")
    return {"stats": {"chunks_pkl": CHUNKS_PKL, "num_chunks": len(state["embedded_chunks"])}}

### 3.6 Store chunks as `(:Chunk)` nodes in Neo4j

In [ ]:
def store_chunks_in_neo4j_node(state: GraphState) -> dict:
    """Create Chunk nodes with text, header, images and the bge-m3 text_embedding."""
    from neo4j import GraphDatabase

    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    upserted = 0
    try:
        with driver.session(database=NEO4J_DATABASE) as s:
            s.run(
                "CREATE VECTOR INDEX chunk_embeddings IF NOT EXISTS "
                "FOR (c:Chunk) ON (c.text_embedding) "
                "OPTIONS {indexConfig: {`vector.dimensions`: $dim, "
                "`vector.similarity_function`: 'cosine'}}",
                dim=VECTOR_DIM,
            ).consume()

            for doc in state["embedded_chunks"]:
                s.run(
                    "MERGE (c:Chunk {chunk_id: $cid}) "
                    "SET c.text = $text, c.header = $header, "
                    "c.images = $images, c.has_images = $has_images, "
                    "c.source = $source, "
                    "c.previous_chunk_id = $prev, c.next_chunk_id = $next, "
                    "c.text_embedding = $vec",
                    cid=doc.metadata["chunk_id"],
                    text=doc.page_content,
                    header=doc.metadata.get("header", ""),
                    images=doc.metadata.get("images", "none"),
                    has_images=bool(doc.metadata.get("has_images", False)),
                    source=doc.metadata.get("source", ""),
                    prev=doc.metadata.get("previous_chunk_id"),
                    next=doc.metadata.get("next_chunk_id"),
                    vec=doc.metadata["embedding"],
                ).consume()
                upserted += 1
    finally:
        driver.close()
    print(f"[store_chunks_in_neo4j] Upserted {upserted} Chunk nodes (with text_embedding).")
    print(f"[store_chunks_in_neo4j] Vector index 'chunk_embeddings' ready ({VECTOR_DIM}-d, cosine).")
    return {"stats": {"chunks_in_neo4j": upserted}}

# 4. Entity & relationship extraction (Ollama)

Custom transformer copied from `graph/langchain_final.ipynb`:
`with_structured_output(EducationalGraph, method="json_schema")` on
`ChatOllama(rnj-1:latest)` with a JSON grammar, plus a manual-JSON fallback for
truncated output.

> **Note:** extracting over all 99 chunks with rnj-1 on CPU can take 1h+. If the
> run is interrupted, re-run with `LOAD_EXISTING_GRAPH_DOCS = True` to resume
> from `extracted_graph_docs.pkl` instead of re-running the LLM.

In [ ]:
from pydantic import BaseModel, Field
from typing import List


class KnowledgeNode(BaseModel):
    id: str = Field(description="The unique name or key identifier of the entity")
    name: str = Field(description="The human-readable name of the entity")
    type: str = Field(
        description="Must be one of: Topic, Subtopic, Concept, Definition, Formula, Theorem, Example"
    )


class KnowledgeRelationship(BaseModel):
    source: str = Field(description="The id of the source node")
    target: str = Field(description="The id of the target node")
    type: str = Field(
        description="Must be one of: COVERS, EXPLAINS, PREREQUISITE_FOR, PART_OF, USES_FORMULA, ILLUSTRATES"
    )


class EducationalGraph(BaseModel):
    nodes: List[KnowledgeNode] = Field(default_factory=list)
    relationships: List[KnowledgeRelationship] = Field(default_factory=list)

print("Pydantic schemas defined: KnowledgeNode / KnowledgeRelationship / EducationalGraph.")

In [ ]:
def extract_entities_node(state: GraphState, context_window: int = CONTEXT_WINDOW) -> dict:
    """Extracts educational nodes and relationships per chunk, returns GraphDocuments."""
    import json
    from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship
    from langchain_ollama import ChatOllama

    llm = ChatOllama(model=EXTRACT_MODEL, temperature=0, num_ctx=context_window, format="json")
    structured_llm = llm.with_structured_output(EducationalGraph, method="json_schema")

    def filter_schema(docs):
        """Drop out-of-schema nodes/relationships; returns (docs, dropped_nodes, dropped_rels)."""
        filtered = []
        dropped_nodes = 0
        dropped_rels = 0
        for g in docs:
            keep_nodes = [n for n in g.nodes if n.type in KG_LABELS]
            keep_ids = {n.id for n in keep_nodes}
            keep_rels = [
                r for r in g.relationships
                if r.type in ALLOWED_RELATIONSHIPS
                and r.source.id in keep_ids
                and r.target.id in keep_ids
            ]
            dropped_nodes += len(g.nodes) - len(keep_nodes)
            dropped_rels += len(g.relationships) - len(keep_rels)
            filtered.append(
                GraphDocument(nodes=keep_nodes, relationships=keep_rels, source=g.source)
            )
        return filtered, dropped_nodes, dropped_rels

    # Optional fast-path: reuse a previously extracted pickle instead of calling Ollama.
    if LOAD_EXISTING_GRAPH_DOCS and os.path.exists(GRAPH_DOCS_PKL):
        with open(GRAPH_DOCS_PKL, "rb") as f:
            docs = pickle.load(f)
        docs, dropped_nodes, dropped_rels = filter_schema(docs)
        print(f"[extract_entities] SKIP Ollama: loaded {len(docs)} GraphDocuments from {GRAPH_DOCS_PKL}")
        if dropped_nodes or dropped_rels:
            print(
                f"[extract_entities] Filtered out {dropped_nodes} out-of-schema "
                f"node(s), {dropped_rels} relationship(s)."
            )
        print("[extract_entities] Set LOAD_EXISTING_GRAPH_DOCS = False to force re-extraction.")
        return {"extracted_graph_docs": docs}

    all_extracted_docs = []
    success_count = 0
    failure_count = 0
    total = len(state["chunks"])
    total_dropped_nodes = 0
    total_dropped_rels = 0

    for idx, chunk in enumerate(state["chunks"]):
        prompt = f"""Extract all relevant educational nodes and relationships from this text.

Allowed Nodes: Topic, Subtopic, Concept, Definition, Formula, Theorem, Example
Allowed Relationships: COVERS, EXPLAINS, PREREQUISITE_FOR, PART_OF, USES_FORMULA, ILLUSTRATES

Text content:
{chunk.page_content}"""

        graph = None
        try:
            result = structured_llm.invoke(prompt)
            graph = result if isinstance(result, EducationalGraph) else EducationalGraph(**result)
        except Exception as e:
            raw_output = None
            try:
                raw_response = llm.invoke(prompt)
                raw_output = getattr(raw_response, "content", None)
            except Exception as e2:
                print(f"[WARN] Raw invoke also failed: {e2}")
            # Retry: parse the raw JSON manually (handles truncated/markdown-wrapped output)
            if raw_output:
                try:
                    raw_text = str(raw_output)
                    json_start = raw_text.find("{")
                    json_end = raw_text.rfind("}")
                    if json_start != -1 and json_end > json_start:
                        graph = EducationalGraph.model_validate(
                            json.loads(raw_text[json_start:json_end + 1])
                        )
                except Exception as e2:
                    print(f"[WARN] Manual JSON retry failed for chunk {idx}: {e2}")

        if graph is None:
            failure_count += 1
            print(f"[extract_entities] chunk {idx}/{total}: FAILED (no parseable JSON)")
            continue

        try:
            langchain_nodes = [
                Node(id=n.id, type=n.type, properties={"name": n.name}) for n in graph.nodes
            ]
            node_dict = {n.id: n for n in langchain_nodes}
            langchain_rels = [
                Relationship(source=node_dict[r.source], target=node_dict[r.target], type=r.type)
                for r in graph.relationships
            ]
            graph_doc = GraphDocument(
                nodes=langchain_nodes, relationships=langchain_rels, source=chunk
            )
            filtered_docs, dn, dr = filter_schema([graph_doc])
            graph_doc = filtered_docs[0]
            all_extracted_docs.append(graph_doc)
            total_dropped_nodes += dn
            total_dropped_rels += dr
            success_count += 1
            extra = f", dropped {dn} out-of-schema node(s)/{dr} rel(s)" if (dn or dr) else ""
            print(
                f"[extract_entities] chunk {idx}/{total}: OK "
                f"({len(langchain_nodes)} nodes, {len(langchain_rels)} rels){extra}"
            )
        except Exception as e:
            failure_count += 1
            print(f"[WARN] Mapping chunk {idx} to GraphDocument failed: {e}")

    with open(GRAPH_DOCS_PKL, "wb") as f:
        pickle.dump(all_extracted_docs, f)

    total_nodes = sum(len(g.nodes) for g in all_extracted_docs)
    total_rels = sum(len(g.relationships) for g in all_extracted_docs)
    print(f"[extract_entities] Extracted {len(all_extracted_docs)} GraphDocuments (ok: {success_count}, failed: {failure_count})")
    print(f"[extract_entities] Total nodes: {total_nodes}, total relationships: {total_rels}")
    if total_dropped_nodes or total_dropped_rels:
        print(
            f"[extract_entities] Filtered out {total_dropped_nodes} out-of-schema "
            f"node(s), {total_dropped_rels} relationship(s)."
        )
    print(f"[extract_entities] Persisted to {GRAPH_DOCS_PKL}")
    return {"extracted_graph_docs": all_extracted_docs}

# 5. Neo4j graph loading + `MENTIONED_IN` chunk linking

In [ ]:
def save_graph_to_neo4j_node(state: GraphState) -> dict:
    """Create entity nodes + relationships via Neo4jGraph.add_graph_documents."""
    from langchain_community.graphs import Neo4jGraph

    graph = Neo4jGraph(
        username=NEO4J_USER,
        password=NEO4J_PASSWORD,
        url=NEO4J_URI,
        database=NEO4J_DATABASE,
    )
    graph.add_graph_documents(state["extracted_graph_docs"], include_source=False)
    print(f"[save_graph_to_neo4j] Loaded {len(state['extracted_graph_docs'])} GraphDocuments into Neo4j.")
    print("[save_graph_to_neo4j] Entity nodes + relationships created (MERGE by node id).")
    return {"stats": {"graph_docs": len(state["extracted_graph_docs"])}}

In [ ]:
def create_mentioned_in_node(state: GraphState) -> dict:
    """Link each entity node to the Chunk nodes that mention it via MENTIONED_IN."""
    from neo4j import GraphDatabase

    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    linked = 0
    skipped = 0
    try:
        with driver.session(database=NEO4J_DATABASE) as s:
            chunk_ids = set(s.run("MATCH (c:Chunk) RETURN c.chunk_id AS cid").value("cid"))
            for g in state["extracted_graph_docs"]:
                cid = chunk_id(g.source.page_content)
                if cid not in chunk_ids:
                    skipped += 1
                    print(f"[create_mentioned_in] WARN: chunk from GraphDocument not found in Neo4j ({cid})")
                    continue
                for n in g.nodes:
                    res = s.run(
                        "MATCH (e {id: $id}) "
                        "MATCH (c:Chunk {chunk_id: $cid}) "
                        "MERGE (e)-[:MENTIONED_IN]->(c)",
                        id=n.id,
                        cid=cid,
                    ).consume()
                    linked += res.counters.relationships_created
    finally:
        driver.close()
    print(f"[create_mentioned_in] Created {linked} MENTIONED_IN relationship(s) (entity -> chunk).")
    return {"stats": {"mentioned_in_created": linked}}

# 6. Pipeline configuration (langgraph StateGraph)

In [ ]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(GraphState)

workflow.add_node("wipe_neo4j", wipe_neo4j_node)
workflow.add_node("load_markdown", load_markdown_node)
workflow.add_node("chunk_markdown", chunk_markdown_node)
workflow.add_node("create_embeddings", create_embeddings_node)
workflow.add_node("save_chunks_to_pkl", save_chunks_to_pkl_node)
workflow.add_node("store_chunks_in_neo4j", store_chunks_in_neo4j_node)
workflow.add_node("extract_entities", extract_entities_node)
workflow.add_node("save_graph_to_neo4j", save_graph_to_neo4j_node)
workflow.add_node("create_mentioned_in", create_mentioned_in_node)

workflow.add_edge(START, "wipe_neo4j")
workflow.add_edge("wipe_neo4j", "load_markdown")
workflow.add_edge("load_markdown", "chunk_markdown")
workflow.add_edge("chunk_markdown", "create_embeddings")
workflow.add_edge("create_embeddings", "save_chunks_to_pkl")
workflow.add_edge("save_chunks_to_pkl", "store_chunks_in_neo4j")
workflow.add_edge("store_chunks_in_neo4j", "extract_entities")
workflow.add_edge("extract_entities", "save_graph_to_neo4j")
workflow.add_edge("save_graph_to_neo4j", "create_mentioned_in")
workflow.add_edge("create_mentioned_in", END)

pipeline = workflow.compile()
print("Pipeline compiled with 9 nodes.")

# 7. Run the pipeline

In [ ]:
config = {"file_path": MD_PATH}
result = pipeline.invoke(config)

print("\n===== PIPELINE COMPLETE =====")
print("Stats:", result.get("stats"))
print("Chunks:", len(result["chunks"]))
print("GraphDocuments:", len(result["extracted_graph_docs"]))

# 8. Verify Neo4j state

In [ ]:
def verify_neo4j():
    from neo4j import GraphDatabase

    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    try:
        with driver.session(database=NEO4J_DATABASE) as s:
            print("\n--- Node counts by label ---")
            for row in s.run("MATCH (n) WITH labels(n) AS lbl, count(*) AS c RETURN lbl, c ORDER BY c DESC"):
                print(f"  {row['lbl']}: {row['c']}")

            print("--- Relationship counts ---")
            for row in s.run("MATCH ()-[r]->() RETURN type(r) AS t, count(*) AS c ORDER BY c DESC"):
                print(f"  {row['t']}: {row['c']}")

            print("--- Indexes (name / type) ---")
            for row in s.run("SHOW INDEXES YIELD name, type"):
                print(f"  {row[0]}: {row[1]}")

            print("--- Sample chunk chain (prev / current / next) ---")
            for row in s.run(
                "MATCH (c:Chunk) "
                "RETURN c.chunk_index AS idx, c.previous_chunk_id AS prev, "
                "c.chunk_id AS cid, c.next_chunk_id AS next "
                "ORDER BY c.chunk_index LIMIT 3"
            ):
                print(f"  chunk {row['idx']}: {row['prev']} <- {row['cid']} -> {row['next']}")

            print("--- Sample Topic -> MENTIONED_IN -> Chunk ---")
            for row in s.run(
                "MATCH (e:Topic)-[:MENTIONED_IN]->(c:Chunk) "
                "RETURN e.id AS eid, e.name AS name, c.chunk_id AS cid, "
                "size(c.text_embedding) AS dim LIMIT 3"
            ):
                print(f"  {row['name']} [{row['eid']}] -> chunk {row['cid']} (embed dim {row['dim']})")
    finally:
        driver.close()


verify_neo4j()